# GAN RICE Cloud Removal: U-Net Generator + ResNet Discriminator

Notebook ini dibuat untuk tugas **cloud removal** pada dataset RICE.

- Input: citra satelit berawan (`cloud`)
- Target: citra satelit bersih (`label`)
- Mask: area awan (`mask`) jika tersedia, terutama pada RICE2
- Generator: U-Net
- Discriminator: ResNet-style conditional discriminator
- Loss: adversarial loss + L1 reconstruction loss + Dice loss
- Evaluasi: TP, FP, TN, FN, IoU, F1/Dice, MAE, MSE, PSNR

In [ ]:
# ============================================================
# 1. Import Library dan Konfigurasi Awal
# ============================================================

from pathlib import Path
import os
import random
import math
from dataclasses import dataclass
from typing import Optional, Tuple, Dict, List

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

In [ ]:
# ============================================================
# 2. Konfigurasi Dataset dan Hyperparameter
# ============================================================

@dataclass
class Config:
    # Ganti ROOT_DIR jika notebook dijalankan dari folder lain.
    ROOT_DIR: Path = Path.cwd()
    DATASET_NAME: str = 'RICE2'  # rekomendasi: RICE2 karena punya mask
    IMAGE_SIZE: int = 512
    BATCH_SIZE: int = 2          # naikkan ke 4 jika GPU cukup
    NUM_WORKERS: int = 0         # Windows/Jupyter lebih aman 0
    VAL_RATIO: float = 0.10
    TEST_RATIO: float = 0.10
    EPOCHS: int = 50
    LR: float = 2e-4
    BETA1: float = 0.5
    BETA2: float = 0.999
    LAMBDA_L1: float = 100.0
    LAMBDA_DICE: float = 10.0
    MASK_THRESHOLD: float = 0.50
    OUTPUT_DIR: Path = Path('outputs_rice_unet_resnet')

CFG = Config()
CFG.OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
print(CFG)

In [ ]:
# ============================================================
# 3. Helper Path Dataset RICE
# ============================================================

def resolve_rice_paths(root_dir: Path, dataset_name: str = 'RICE2') -> Dict[str, Optional[Path]]:
    """Mengembalikan path folder cloud, label, mask, dan test untuk RICE1/RICE2."""
    rice_root = root_dir / 'RICE' / dataset_name
    if not rice_root.exists():
        raise FileNotFoundError(f'Folder dataset tidak ditemukan: {rice_root}')

    if dataset_name.upper() == 'RICE2':
        paths = {
            'train_cloud': rice_root / 'cloud',
            'train_label': rice_root / 'label',
            'train_mask': rice_root / 'mask',
            'test_cloud': rice_root / 'Test' / 'Cloud',
            'test_label': rice_root / 'Test' / 'Label',
            'test_mask': None,
        }
    elif dataset_name.upper() == 'RICE1':
        paths = {
            'train_cloud': rice_root / 'cloud',
            'train_label': rice_root / 'label',
            'train_mask': None,
            'test_cloud': rice_root / 'Test' / 'cloud',
            'test_label': rice_root / 'Test' / 'label',
            'test_mask': None,
        }
    else:
        raise ValueError("dataset_name harus 'RICE1' atau 'RICE2'")

    for key, path in paths.items():
        if path is not None and not path.exists():
            raise FileNotFoundError(f'Path {key} tidak ditemukan: {path}')
    return paths

paths = resolve_rice_paths(CFG.ROOT_DIR, CFG.DATASET_NAME)
paths

In [ ]:
# ============================================================
# 4. Dataset Class
# ============================================================

IMG_EXTS = {'.png', '.jpg', '.jpeg', '.tif', '.tiff'}

def list_image_files(folder: Path) -> List[Path]:
    return sorted([p for p in folder.iterdir() if p.suffix.lower() in IMG_EXTS])

def make_pseudo_mask(cloud_tensor: torch.Tensor, clear_tensor: torch.Tensor, threshold: float = 0.15) -> torch.Tensor:
    """Fallback mask jika dataset tidak punya mask.

    Input cloud/clear dalam range [-1, 1]. Mask dibuat dari rata-rata absolute difference.
    Output shape: [1, H, W], nilai 0/1.
    """
    diff = torch.abs(cloud_tensor - clear_tensor).mean(dim=0, keepdim=True)
    return (diff > threshold).float()

class RICECloudRemovalDataset(Dataset):
    def __init__(self, cloud_dir: Path, label_dir: Path, mask_dir: Optional[Path] = None, image_size: int = 512):
        self.cloud_dir = cloud_dir
        self.label_dir = label_dir
        self.mask_dir = mask_dir
        self.image_size = image_size

        cloud_files = list_image_files(cloud_dir)
        label_files = list_image_files(label_dir)
        label_names = {p.name for p in label_files}

        self.samples = []
        for cloud_path in cloud_files:
            label_path = label_dir / cloud_path.name
            if label_path.exists():
                mask_path = mask_dir / cloud_path.name if mask_dir is not None else None
                if mask_path is not None and not mask_path.exists():
                    mask_path = None
                self.samples.append((cloud_path, label_path, mask_path))

        if not self.samples:
            raise ValueError(f'Tidak ada pasangan cloud-label cocok di {cloud_dir} dan {label_dir}')

        self.image_transform = transforms.Compose([
            transforms.Resize((image_size, image_size), interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
        ])
        self.mask_transform = transforms.Compose([
            transforms.Resize((image_size, image_size), interpolation=transforms.InterpolationMode.NEAREST),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        cloud_path, label_path, mask_path = self.samples[idx]
        cloud = Image.open(cloud_path).convert('RGB')
        label = Image.open(label_path).convert('RGB')

        cloud_tensor = self.image_transform(cloud)
        label_tensor = self.image_transform(label)

        if mask_path is not None:
            mask_img = Image.open(mask_path).convert('L')
            mask_tensor = self.mask_transform(mask_img)
            mask_tensor = (mask_tensor > CFG.MASK_THRESHOLD).float()
        else:
            mask_tensor = make_pseudo_mask(cloud_tensor, label_tensor)

        return {
            'cloud': cloud_tensor,
            'label': label_tensor,
            'mask': mask_tensor,
            'name': cloud_path.name,
        }

train_full_dataset = RICECloudRemovalDataset(
    cloud_dir=paths['train_cloud'],
    label_dir=paths['train_label'],
    mask_dir=paths['train_mask'],
    image_size=CFG.IMAGE_SIZE,
)

print('Total train/full samples:', len(train_full_dataset))
print('Contoh sample keys:', train_full_dataset[0].keys())
print('Cloud shape:', train_full_dataset[0]['cloud'].shape)
print('Label shape:', train_full_dataset[0]['label'].shape)
print('Mask shape:', train_full_dataset[0]['mask'].shape)

In [ ]:
# ============================================================
# 5. Split Data dan DataLoader
# ============================================================

n_total = len(train_full_dataset)
n_test = int(n_total * CFG.TEST_RATIO)
n_val = int(n_total * CFG.VAL_RATIO)
n_train = n_total - n_val - n_test

train_dataset, val_dataset, test_dataset = random_split(
    train_full_dataset,
    [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(SEED),
)

train_loader = DataLoader(train_dataset, batch_size=CFG.BATCH_SIZE, shuffle=True, num_workers=CFG.NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=CFG.BATCH_SIZE, shuffle=False, num_workers=CFG.NUM_WORKERS)
test_loader = DataLoader(test_dataset, batch_size=CFG.BATCH_SIZE, shuffle=False, num_workers=CFG.NUM_WORKERS)

print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')

In [ ]:
# ============================================================
# 6. Visualisasi Data
# ============================================================

def denormalize_img(tensor: torch.Tensor) -> torch.Tensor:
    return (tensor * 0.5 + 0.5).clamp(0, 1)

def tensor_to_numpy_img(tensor: torch.Tensor) -> np.ndarray:
    return denormalize_img(tensor.detach().cpu()).permute(1, 2, 0).numpy()

def show_sample(sample):
    cloud = tensor_to_numpy_img(sample['cloud'])
    label = tensor_to_numpy_img(sample['label'])
    mask = sample['mask'].squeeze().detach().cpu().numpy()

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(cloud)
    axes[0].set_title('Input Cloudy')
    axes[0].axis('off')
    axes[1].imshow(label)
    axes[1].set_title('Ground Truth Clear')
    axes[1].axis('off')
    axes[2].imshow(mask, cmap='gray')
    axes[2].set_title('Cloud Mask')
    axes[2].axis('off')
    plt.show()

show_sample(train_full_dataset[0])

In [ ]:
# ============================================================
# 7. U-Net Generator
# ============================================================

class DownBlock(nn.Module):
    def __init__(self, in_channels, out_channels, apply_batchnorm=True):
        super().__init__()
        layers = [nn.Conv2d(in_channels, out_channels, kernel_size=4, stride=2, padding=1, bias=False)]
        if apply_batchnorm:
            layers.append(nn.BatchNorm2d(out_channels))
        layers.append(nn.LeakyReLU(0.2, inplace=True))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)

class UpBlock(nn.Module):
    def __init__(self, in_channels, out_channels, apply_dropout=False):
        super().__init__()
        layers = [
            nn.ConvTranspose2d(in_channels, out_channels, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
        ]
        if apply_dropout:
            layers.append(nn.Dropout(0.5))
        layers.append(nn.ReLU(inplace=True))
        self.block = nn.Sequential(*layers)

    def forward(self, x, skip):
        x = self.block(x)
        return torch.cat([x, skip], dim=1)

class UNetGenerator(nn.Module):
    """U-Net generator untuk input/output 512x512 RGB."""
    def __init__(self, in_channels=3, out_channels=3):
        super().__init__()
        self.down1 = DownBlock(in_channels, 64, apply_batchnorm=False)   # 512 -> 256
        self.down2 = DownBlock(64, 128)                                  # 256 -> 128
        self.down3 = DownBlock(128, 256)                                 # 128 -> 64
        self.down4 = DownBlock(256, 512)                                 # 64 -> 32
        self.down5 = DownBlock(512, 512)                                 # 32 -> 16
        self.down6 = DownBlock(512, 512)                                 # 16 -> 8
        self.down7 = DownBlock(512, 512)                                 # 8 -> 4
        self.down8 = DownBlock(512, 512, apply_batchnorm=False)          # 4 -> 2

        self.up1 = UpBlock(512, 512, apply_dropout=True)                 # 2 -> 4
        self.up2 = UpBlock(1024, 512, apply_dropout=True)                # 4 -> 8
        self.up3 = UpBlock(1024, 512, apply_dropout=True)                # 8 -> 16
        self.up4 = UpBlock(1024, 512)                                    # 16 -> 32
        self.up5 = UpBlock(1024, 256)                                    # 32 -> 64
        self.up6 = UpBlock(512, 128)                                     # 64 -> 128
        self.up7 = UpBlock(256, 64)                                      # 128 -> 256

        self.final = nn.Sequential(
            nn.ConvTranspose2d(128, out_channels, kernel_size=4, stride=2, padding=1),
            nn.Tanh(),
        )                                                               # 256 -> 512

    def forward(self, x):
        d1 = self.down1(x)
        d2 = self.down2(d1)
        d3 = self.down3(d2)
        d4 = self.down4(d3)
        d5 = self.down5(d4)
        d6 = self.down6(d5)
        d7 = self.down7(d6)
        d8 = self.down8(d7)

        u1 = self.up1(d8, d7)
        u2 = self.up2(u1, d6)
        u3 = self.up3(u2, d5)
        u4 = self.up4(u3, d4)
        u5 = self.up5(u4, d3)
        u6 = self.up6(u5, d2)
        u7 = self.up7(u6, d1)
        return self.final(u7)

In [ ]:
# ============================================================
# 8. ResNet-style Conditional Discriminator
# ============================================================

class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(channels, channels, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(channels),
        )
        self.act = nn.LeakyReLU(0.2, inplace=True)

    def forward(self, x):
        return self.act(x + self.block(x))

class ResDownBlock(nn.Module):
    def __init__(self, in_channels, out_channels, apply_batchnorm=True):
        super().__init__()
        main_layers = [nn.Conv2d(in_channels, out_channels, kernel_size=4, stride=2, padding=1, bias=False)]
        if apply_batchnorm:
            main_layers.append(nn.BatchNorm2d(out_channels))
        main_layers.append(nn.LeakyReLU(0.2, inplace=True))
        self.down = nn.Sequential(*main_layers)
        self.res = ResidualBlock(out_channels)

    def forward(self, x):
        return self.res(self.down(x))

class ResNetDiscriminator(nn.Module):
    """Conditional discriminator berbasis residual block.

    Input discriminator adalah concat cloudy dan target/fake:
    [B, 3, H, W] + [B, 3, H, W] -> [B, 6, H, W].
    Output berupa satu logit real/fake per gambar.
    """
    def __init__(self, in_channels=6):
        super().__init__()
        self.features = nn.Sequential(
            ResDownBlock(in_channels, 64, apply_batchnorm=False),   # 512 -> 256
            ResDownBlock(64, 128),                                  # 256 -> 128
            ResDownBlock(128, 256),                                 # 128 -> 64
            ResDownBlock(256, 512),                                 # 64 -> 32
            ResDownBlock(512, 512),                                 # 32 -> 16
            nn.AdaptiveAvgPool2d(1),
        )
        self.classifier = nn.Linear(512, 1)

    def forward(self, cloudy, target):
        x = torch.cat([cloudy, target], dim=1)
        feat = self.features(x).flatten(1)
        return self.classifier(feat)

In [ ]:
# ============================================================
# 9. Loss: GAN Loss + L1 Loss + Dice Loss
# ============================================================

criterion_gan = nn.BCEWithLogitsLoss()
criterion_l1 = nn.L1Loss()

def soft_dice_loss(pred_mask: torch.Tensor, true_mask: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    """Dice loss untuk mask probabilistik.

    pred_mask dan true_mask shape [B, 1, H, W], range [0, 1].
    """
    pred = pred_mask.contiguous().view(pred_mask.size(0), -1)
    true = true_mask.contiguous().view(true_mask.size(0), -1)
    intersection = (pred * true).sum(dim=1)
    denominator = pred.sum(dim=1) + true.sum(dim=1)
    dice = (2.0 * intersection + eps) / (denominator + eps)
    return 1.0 - dice.mean()

def predicted_restoration_mask(cloud: torch.Tensor, generated: torch.Tensor) -> torch.Tensor:
    """Mask prediksi area yang diubah/restorasi oleh generator.

    True mask pada RICE2 menunjukkan area awan pada input. Karena generator tidak punya output
    segmentasi eksplisit, mask prediksi dibuat dari seberapa besar perubahan antara citra
    cloudy dan hasil generated. Area yang banyak berubah dianggap area yang direstorasi.
    Nilai soft menjaga Dice loss tetap differentiable.
    """
    change = torch.abs(cloud - generated).mean(dim=1, keepdim=True)  # range kira-kira [0, 2]
    return torch.clamp(change / 2.0, 0.0, 1.0)

In [ ]:
# ============================================================
# 10. Metrik Evaluasi Pixel-wise
# ============================================================

@torch.no_grad()
def compute_confusion_from_masks(pred_mask: torch.Tensor, true_mask: torch.Tensor, threshold: float = 0.5) -> Dict[str, int]:
    pred = (pred_mask >= threshold).bool()
    true = (true_mask >= threshold).bool()
    tp = torch.logical_and(pred, true).sum().item()
    fp = torch.logical_and(pred, ~true).sum().item()
    tn = torch.logical_and(~pred, ~true).sum().item()
    fn = torch.logical_and(~pred, true).sum().item()
    return {'TP': int(tp), 'FP': int(fp), 'TN': int(tn), 'FN': int(fn)}

def metrics_from_confusion(conf: Dict[str, int], eps: float = 1e-8) -> Dict[str, float]:
    tp, fp, tn, fn = conf['TP'], conf['FP'], conf['TN'], conf['FN']
    iou = tp / (tp + fp + fn + eps)
    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)
    f1 = 2 * precision * recall / (precision + recall + eps)
    accuracy = (tp + tn) / (tp + fp + tn + fn + eps)
    return {
        'IoU': iou,
        'Precision': precision,
        'Recall': recall,
        'F1_Dice': f1,
        'Accuracy': accuracy,
    }

@torch.no_grad()
def image_quality_metrics(generated: torch.Tensor, label: torch.Tensor, eps: float = 1e-8) -> Dict[str, float]:
    gen01 = denormalize_img(generated)
    lab01 = denormalize_img(label)
    mae = torch.mean(torch.abs(gen01 - lab01)).item()
    mse = torch.mean((gen01 - lab01) ** 2).item()
    psnr = 10.0 * math.log10(1.0 / (mse + eps))
    return {'MAE': mae, 'MSE': mse, 'PSNR': psnr}

In [ ]:
# ============================================================
# 11. Inisialisasi Model dan Optimizer
# ============================================================

generator = UNetGenerator().to(DEVICE)
discriminator = ResNetDiscriminator().to(DEVICE)

optimizer_G = torch.optim.Adam(generator.parameters(), lr=CFG.LR, betas=(CFG.BETA1, CFG.BETA2))
optimizer_D = torch.optim.Adam(discriminator.parameters(), lr=CFG.LR, betas=(CFG.BETA1, CFG.BETA2))

print('Generator parameters:', sum(p.numel() for p in generator.parameters()))
print('Discriminator parameters:', sum(p.numel() for p in discriminator.parameters()))

In [ ]:
# ============================================================
# 12. Sanity Check Forward Pass
# ============================================================

batch = next(iter(train_loader))
cloud = batch['cloud'].to(DEVICE)
label = batch['label'].to(DEVICE)
mask = batch['mask'].to(DEVICE)

with torch.no_grad():
    fake = generator(cloud)
    pred_real = discriminator(cloud, label)
    pred_fake = discriminator(cloud, fake)
    fake_restoration_mask = predicted_restoration_mask(cloud, fake)

print('cloud:', cloud.shape)
print('label:', label.shape)
print('mask:', mask.shape)
print('fake:', fake.shape)
print('D(real):', pred_real.shape)
print('D(fake):', pred_fake.shape)
print('dice loss example:', soft_dice_loss(fake_restoration_mask, mask).item())

In [ ]:
# ============================================================
# 13. Training dan Validation Function
# ============================================================

def train_one_epoch(epoch: int):
    generator.train()
    discriminator.train()

    logs = {'loss_D': [], 'loss_G': [], 'loss_GAN': [], 'loss_L1': [], 'loss_Dice': []}

    for batch_idx, batch in enumerate(train_loader):
        cloud = batch['cloud'].to(DEVICE)
        label = batch['label'].to(DEVICE)
        mask = batch['mask'].to(DEVICE)
        batch_size = cloud.size(0)

        real_targets = torch.ones((batch_size, 1), device=DEVICE)
        fake_targets = torch.zeros((batch_size, 1), device=DEVICE)

        # -------------------------
        # Train Discriminator
        # -------------------------
        optimizer_D.zero_grad(set_to_none=True)

        fake = generator(cloud)
        pred_real = discriminator(cloud, label)
        pred_fake = discriminator(cloud, fake.detach())

        loss_D_real = criterion_gan(pred_real, real_targets)
        loss_D_fake = criterion_gan(pred_fake, fake_targets)
        loss_D = 0.5 * (loss_D_real + loss_D_fake)
        loss_D.backward()
        optimizer_D.step()

        # -------------------------
        # Train Generator
        # -------------------------
        optimizer_G.zero_grad(set_to_none=True)

        pred_fake_for_G = discriminator(cloud, fake)
        loss_GAN = criterion_gan(pred_fake_for_G, real_targets)
        loss_L1 = criterion_l1(fake, label)
        pred_restoration_mask = predicted_restoration_mask(cloud, fake)
        loss_Dice = soft_dice_loss(pred_restoration_mask, mask)

        loss_G = loss_GAN + CFG.LAMBDA_L1 * loss_L1 + CFG.LAMBDA_DICE * loss_Dice
        loss_G.backward()
        optimizer_G.step()

        logs['loss_D'].append(loss_D.item())
        logs['loss_G'].append(loss_G.item())
        logs['loss_GAN'].append(loss_GAN.item())
        logs['loss_L1'].append(loss_L1.item())
        logs['loss_Dice'].append(loss_Dice.item())

    return {k: float(np.mean(v)) for k, v in logs.items()}

@torch.no_grad()
def evaluate(loader):
    generator.eval()

    total_conf = {'TP': 0, 'FP': 0, 'TN': 0, 'FN': 0}
    quality_logs = {'MAE': [], 'MSE': [], 'PSNR': []}
    dice_losses = []

    for batch in loader:
        cloud = batch['cloud'].to(DEVICE)
        label = batch['label'].to(DEVICE)
        mask = batch['mask'].to(DEVICE)

        fake = generator(cloud)
        pred_mask = predicted_restoration_mask(cloud, fake)

        conf = compute_confusion_from_masks(pred_mask.cpu(), mask.cpu(), threshold=CFG.MASK_THRESHOLD)
        for key in total_conf:
            total_conf[key] += conf[key]

        q = image_quality_metrics(fake.cpu(), label.cpu())
        for key in quality_logs:
            quality_logs[key].append(q[key])

        dice_losses.append(soft_dice_loss(pred_mask, mask).item())

    pixel_metrics = metrics_from_confusion(total_conf)
    quality_metrics = {k: float(np.mean(v)) for k, v in quality_logs.items()}
    return {**total_conf, **pixel_metrics, **quality_metrics, 'DiceLoss': float(np.mean(dice_losses))}

In [ ]:
# ============================================================
# 14. Visualisasi Hasil Prediksi
# ============================================================

@torch.no_grad()
def show_predictions(loader, max_items: int = 3):
    generator.eval()
    batch = next(iter(loader))
    cloud = batch['cloud'].to(DEVICE)
    label = batch['label'].to(DEVICE)
    mask = batch['mask']
    fake = generator(cloud).cpu()

    n = min(max_items, cloud.size(0))
    fig, axes = plt.subplots(n, 4, figsize=(18, 4 * n))
    if n == 1:
        axes = np.expand_dims(axes, axis=0)

    for i in range(n):
        axes[i, 0].imshow(tensor_to_numpy_img(cloud[i].cpu()))
        axes[i, 0].set_title('Input Cloudy')
        axes[i, 0].axis('off')

        axes[i, 1].imshow(tensor_to_numpy_img(fake[i]))
        axes[i, 1].set_title('Generated Clear')
        axes[i, 1].axis('off')

        axes[i, 2].imshow(tensor_to_numpy_img(label[i].cpu()))
        axes[i, 2].set_title('Ground Truth')
        axes[i, 2].axis('off')

        axes[i, 3].imshow(mask[i].squeeze().numpy(), cmap='gray')
        axes[i, 3].set_title('Cloud Mask')
        axes[i, 3].axis('off')

    plt.tight_layout()
    plt.show()

show_predictions(val_loader, max_items=2)

In [ ]:
# ============================================================
# 15. Training Loop
# ============================================================

history = []
best_val_f1 = -1.0

for epoch in range(1, CFG.EPOCHS + 1):
    train_logs = train_one_epoch(epoch)
    val_logs = evaluate(val_loader)

    row = {'epoch': epoch, **train_logs, **{f'val_{k}': v for k, v in val_logs.items()}}
    history.append(row)

    print(
        f"Epoch {epoch:03d}/{CFG.EPOCHS} | "
        f"D: {train_logs['loss_D']:.4f} | "
        f"G: {train_logs['loss_G']:.4f} | "
        f"L1: {train_logs['loss_L1']:.4f} | "
        f"DiceLoss: {train_logs['loss_Dice']:.4f} | "
        f"Val IoU: {val_logs['IoU']:.4f} | "
        f"Val F1/Dice: {val_logs['F1_Dice']:.4f} | "
        f"Val PSNR: {val_logs['PSNR']:.2f}"
    )

    if val_logs['F1_Dice'] > best_val_f1:
        best_val_f1 = val_logs['F1_Dice']
        torch.save(generator.state_dict(), CFG.OUTPUT_DIR / 'best_generator_unet_resnet_rice.pth')
        torch.save(discriminator.state_dict(), CFG.OUTPUT_DIR / 'best_discriminator_resnet_rice.pth')

    if epoch % 5 == 0:
        show_predictions(val_loader, max_items=2)

    if epoch % 10 == 0:
        checkpoint = {
            'epoch': epoch,
            'generator_state_dict': generator.state_dict(),
            'discriminator_state_dict': discriminator.state_dict(),
            'optimizer_G_state_dict': optimizer_G.state_dict(),
            'optimizer_D_state_dict': optimizer_D.state_dict(),
            'config': CFG.__dict__,
            'history': history,
        }
        torch.save(checkpoint, CFG.OUTPUT_DIR / f'checkpoint_epoch_{epoch}.pth')

history_df = pd.DataFrame(history)
history_df.to_csv(CFG.OUTPUT_DIR / 'training_history.csv', index=False)
history_df.tail()

In [ ]:
# ============================================================
# 16. Evaluasi Akhir pada Test Split
# ============================================================

best_path = CFG.OUTPUT_DIR / 'best_generator_unet_resnet_rice.pth'
if best_path.exists():
    generator.load_state_dict(torch.load(best_path, map_location=DEVICE))
    print('Loaded best generator:', best_path)

test_metrics = evaluate(test_loader)
print('Test Metrics:')
for key, value in test_metrics.items():
    print(f'{key}: {value}')

pd.DataFrame([test_metrics]).to_csv(CFG.OUTPUT_DIR / 'test_metrics.csv', index=False)
show_predictions(test_loader, max_items=3)

In [ ]:
# ============================================================
# 17. Inference pada Folder Test Asli RICE Jika Tersedia
# ============================================================

class RICETestFolderDataset(Dataset):
    def __init__(self, cloud_dir: Path, label_dir: Optional[Path] = None, image_size: int = 512):
        self.cloud_files = list_image_files(cloud_dir)
        self.label_dir = label_dir
        self.image_transform = transforms.Compose([
            transforms.Resize((image_size, image_size), interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
        ])

    def __len__(self):
        return len(self.cloud_files)

    def __getitem__(self, idx):
        cloud_path = self.cloud_files[idx]
        cloud = self.image_transform(Image.open(cloud_path).convert('RGB'))
        item = {'cloud': cloud, 'name': cloud_path.name}
        if self.label_dir is not None:
            label_path = self.label_dir / cloud_path.name
            if label_path.exists():
                item['label'] = self.image_transform(Image.open(label_path).convert('RGB'))
        return item

@torch.no_grad()
def save_rice_folder_predictions():
    if paths['test_cloud'] is None or not paths['test_cloud'].exists():
        print('Folder test asli tidak tersedia.')
        return

    out_dir = CFG.OUTPUT_DIR / 'rice_original_test_predictions'
    out_dir.mkdir(exist_ok=True, parents=True)

    dataset = RICETestFolderDataset(paths['test_cloud'], paths['test_label'], CFG.IMAGE_SIZE)
    loader = DataLoader(dataset, batch_size=1, shuffle=False, num_workers=0)

    generator.eval()
    for batch in loader:
        cloud = batch['cloud'].to(DEVICE)
        fake = generator(cloud)[0].cpu()
        img = (denormalize_img(fake).permute(1, 2, 0).numpy() * 255).astype(np.uint8)
        Image.fromarray(img).save(out_dir / batch['name'][0])

    print('Prediksi tersimpan di:', out_dir)

save_rice_folder_predictions()

In [ ]:
# ============================================================
# 18. Plot Training History
# ============================================================

if 'history_df' in globals() and not history_df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].plot(history_df['epoch'], history_df['loss_D'], label='D Loss')
    axes[0].plot(history_df['epoch'], history_df['loss_G'], label='G Loss')
    axes[0].set_title('GAN Loss')
    axes[0].legend()

    axes[1].plot(history_df['epoch'], history_df['val_IoU'], label='Val IoU')
    axes[1].plot(history_df['epoch'], history_df['val_F1_Dice'], label='Val F1/Dice')
    axes[1].set_title('Pixel-wise Metrics')
    axes[1].legend()

    axes[2].plot(history_df['epoch'], history_df['val_PSNR'], label='Val PSNR')
    axes[2].set_title('Image Quality')
    axes[2].legend()

    plt.tight_layout()
    plt.show()
else:
    print('History belum tersedia. Jalankan training loop terlebih dahulu.')